In [6]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt

def load_all_logs(log_root_dir):
    all_files = glob.glob(os.path.join(log_root_dir, "**", "*.csv"), recursive=True)
    
    log_dfs = []
    for file in all_files:
        df = pd.read_csv(file)
        log_dfs.append(df)

    logs = pd.concat(log_dfs, ignore_index=True)
    return logs

In [ ]:
def attack_counts(k, flow_csv, log_root_dir):
    print(f"Labeling {flow_csv} using logs in {log_root_dir}...")
    flows = pd.read_csv(flow_csv) #time_window already k-times timestamp from prev logs
    logs = load_all_logs(log_root_dir)

    

    logs["timestamp"] = pd.to_datetime(logs["Timestamp"], utc=True, errors="coerce")
    logs = logs.dropna(subset=["timestamp"])

    logs["epoch"] = logs["timestamp"].apply(lambda x: x.timestamp())
    logs["time_window"] = (logs["epoch"] * k).astype(int)   # 1000/k ms windows

    start, end = flows["time_window"].min(), flows["time_window"].max() 

    logs_in_range = logs[(logs["time_window"] >= start) & (logs["time_window"] <= end)]

    logs_grouped = (
        logs_in_range.groupby(["time_window", "TargetIP"])
        .agg(attack_count=("Attack", "count"))
        .reset_index()
    )

    flows["time_window"] = flows["time_window"].astype(int)

    # common = set(zip(flows["time_window"], flows["ip.dst"])) & set(zip(logs_in_range["time_window"], logs_in_range["TargetIP"]))
    # print(len(common))
    
    merged = flows.merge(
        logs_grouped,
        left_on=["time_window", "ip.dst"],
        right_on=["time_window", "TargetIP"],
        how="left"
    )

    merged["attack_count"] = merged["attack_count"].fillna(0)

    merged["is_attack"] = (merged["attack_count"] > 0).astype(int)

    merged.drop(columns=["TargetIP"], inplace=True)

    print("Attack windows:", merged["is_attack"].sum())
    print("Windows from logs:", logs_in_range['time_window'].nunique())



In [8]:
attack_counts(1, "../train/1s_cscada_flows.csv", "../data/attack/compromised-scada/attack logs")

Labeling ../train/1s_cscada_flows.csv using logs in ../data/attack/compromised-scada/attack logs...
Attack windows: 919
Total counted attacks: 146427.0
Windows from logs: 3912


In [9]:
attack_counts(10, "../train/100ms_cscada_flows.csv", "../data/attack/compromised-scada/attack logs")
# I think there is a small gap between the log and the actual pcap, and both are falling in different windows

Labeling ../train/100ms_cscada_flows.csv using logs in ../data/attack/compromised-scada/attack logs...
Attack windows: 242
Total counted attacks: 26269.0
Windows from logs: 5573
